# Current version : 11.B (2025-11-14)

# Libraries and directory (always run)

In [ ]:
### import necessary libraries
# import anndata as ad
from datetime import datetime
import os
from IPython.display import display
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import pandas as pd
import geopandas as gpd
import seaborn as sns
import scanpy as sc
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
import warnings
from module.config_local import dir_processed

warnings.filterwarnings("ignore") 
sc.logging.print_header()
sc.set_figure_params(facecolor="white", figsize=(8, 8))
sc.settings.verbosity = 1 # errors (0), warnings (1), info (2), hints (3)
plt.rcParams["font.family"] = "Arial"
sns.set_style("white")

import matplotlib as mpl
mpl.rcParams['svg.fonttype'] = 'none'

pd.options.display.max_rows = 9999

start_time = datetime.now()

def print_with_elapsed_time(message):
    elapsed_time = datetime.now() - start_time
    elapsed_seconds = elapsed_time.total_seconds()
    print(f"[{elapsed_seconds:.2f} seconds] {message}")

In [ ]:
# print(f"geopandas version: {gpd.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"scanpy version: {sc.__version__}")

In [ ]:
from module.config_local import sample_name_import

name_dir = ''

samples_ids = sample_name_import(name_dir)

print(len(samples_ids))
print(samples_ids)

# Data import

In [ ]:
adata = sc.read_h5ad(f"{dir_processed}/h5ad/{name_dir}/{name_dir}_clusters.h5ad.gz")

# adata = sc.read_h5ad(f"{dir_processed}/h5ad/{name_dir}/{name_dir}_final.h5ad.gz")

In [ ]:
df = pd.read_parquet(f"{dir_processed}/csv/{name_dir}/{name_dir}_norm_combined.parquet")

# Annotations (MapMyCell)

## Initial annotation

### Import MMC

In [ ]:
from module.xenium_preprocessing import mmc_merge
adata = mmc_merge(adata, dir_processed, name_dir)

In [ ]:
adata.obs['mmc:subclass_name'].isna().sum()

# Should be 0. If not zero, some cell are not annotated.
# Check the name of the CorrelationMapping files; if there is any missing, etc.

In [ ]:
adata.obs.sample(10)

### Automatic initial annotation

In [ ]:
from module.subclustering_Xe import automatic_initial_annotation

adata = automatic_initial_annotation(adata, 'leiden')

## Cluster check

In [ ]:
# adata.obs['mmc:class_name'].value_counts().sort_values().to_csv('data/mmc_cellnb_class.csv')
adata.obs['cell_type_auto'].value_counts().sort_values()


In [ ]:
import module.dataviz_analysis as da

da.cluster_plot(adata,
                name_dir,
                cluster_to_use="cell_type_newnum_auto",
                cluster_to_map=['all'],
                )



In [ ]:
## to move


# To rename a cell type in case of typo or mistakes (easier than re-running)
rename_subclass = {

}
                  
adata.obs['cell_type_final'] = adata.obs['cell_type_final'].apply(lambda x: rename_subclass[x] if x in rename_subclass else x)
all_cell_type = adata.obs['cell_type_final'].unique()
list_cell_nb = range(0, len(all_cell_type))
mapping_dict = dict(zip(all_cell_type,list_cell_nb))
adata.obs['cell_type_newnum_final'] = adata.obs['cell_type_final'].map(mapping_dict)
adata.obs['cell_type_final'].unique()

In [ ]:
adata = adata[~(adata.obs['cell_type_final']=="Undefined")]

In [ ]:
if 'leiden_colors' in adata.obs:
    adata.obs = adata.obs.drop(columns=['leiden_colors'])

# adata.write(f"{dir_processed}/h5ad/{name_dir}/{name_dir}_MMC_Banksy_clusters.h5ad.gz", compression='gzip')
adata.write(f"{dir_processed}/h5ad/{name_dir}/{name_dir}_clusters.h5ad.gz", compression='gzip')

In [ ]:
adata = sc.read_h5ad(f"{dir_processed}/h5ad/{name_dir}/{name_dir}_clusters.h5ad.gz")

## Subclustering

In [ ]:
adata.obs['cell_type_auto'].value_counts()
# adata.obs['cell_type_final'].value_counts()

### Automatic subclustering

In [ ]:
from module.subclustering_Xe import auto_subclustering2

auto_subclustering2(adata_to_sub = adata, ###
                    all_types =  'all', ### 'all' or ['0','1'] for selected number of clusters
                    Clusters_to_use = 'cell_type_newnum_auto',
                    resolution = 0.2)

#### Table

In [ ]:
from module.subclustering_Xe import cluster_table

cont_tab, cont_tab_sub, cluster_df = cluster_table(adata_to_use = adata,
                                                   Clusters_to_use = 'cell_type_newnum_auto_sub',
                                                   sort_order='Celltype',
                                                   sort_ascend = True
                                                   )

cluster_df

In [ ]:
da.cluster_plot(adata,
             name_dir=name_dir,
             dir_processed=dir_processed,
             cluster_to_use='cell_type_newnum_auto_sub',
             cluster_to_map=['all'])

In [ ]:
tst_dict = cont_tab_sub.T.idxmax(axis=0).to_dict()
tst_dict = dict(zip(cluster_df.index, cluster_df['Celltype']))
tst_dict

#### Final clusters

In [ ]:
# Final clusters

## Copy dictionnary from previous cell to rename cell types names
rename_subclass = {
    
}

adata.obs['cell_type_final'] = adata.obs['cell_type_newnum_auto_sub'].apply(lambda x: rename_subclass[x] if x in rename_subclass else x)
adata.obs['cell_type_final'].unique()

all_cell_type = adata.obs['cell_type_final'].unique()
list_cell_nb = range(0, len(all_cell_type))
mapping_dict = dict(zip(all_cell_type,list_cell_nb))
adata.obs['cell_type_newnum_final'] = adata.obs['cell_type_final'].map(mapping_dict)
mapping_dict

In [ ]:
### To rename only some cell types
rename_subclass = dict(zip(adata.obs['cell_type_newnum_final'],adata.obs['cell_type_final']))
rename_subclass_temp = {
# 43 : "OB STR CTX IMN", ## Example

}
rename_subclass.update(rename_subclass_temp)
adata.obs['cell_type_final'] = adata.obs['cell_type_newnum_final'].apply(lambda x: rename_subclass[x] if x in rename_subclass else x)
adata.obs['cell_type_final'].unique()

all_cell_type = adata.obs['cell_type_final'].unique()
list_cell_nb = range(0, len(all_cell_type))
mapping_dict = dict(zip(all_cell_type,list_cell_nb))
adata.obs['cell_type_newnum_final'] = adata.obs['cell_type_final'].map(mapping_dict)
mapping_dict

In [ ]:
### Correlation map
# Clusters_to_use = 'cell_type_newnum_auto_sub'
# cont_tab = pd.crosstab(adata_filter.obs[Clusters_to_use], adata_filter.obs['mmc:class_name'], normalize="index")
# cont_tab = cont_tab.loc[:, cont_tab.sum(axis=0) > 0.1] 
plt.figure(figsize=(40, 20))
sns.heatmap(cont_tab.T, annot=True, cmap="YlGnBu", fmt=".1f", cbar = False)

In [ ]:
### Correlation map
# cont_tab_sub = pd.crosstab(adata_filter.obs[Clusters_to_use],adata_filter.obs['mmc:subclass_name'], normalize="index")
# cont_tab_sub = cont_tab_sub.loc[:, cont_tab_sub.sum(axis=0) > 0.05] 
plt.figure(figsize=(40, 50))
sns.heatmap(cont_tab_sub.T, annot=True, cmap="YlGnBu", fmt=".1f", cbar = False)

In [ ]:
cont_tab_sub.T[16].sort_values(ascending=False).head()

### Manual subclustering

In [ ]:
# Select cluster to use for manual subclustering
clusters_to_use = 'cell_type_newnum_final'

In [ ]:
### Correlation map
cont_tab = pd.crosstab(adata.obs[clusters_to_use], adata.obs['mmc:subclass_name'], normalize="index")
cont_tab = cont_tab.loc[:, cont_tab.sum(axis=0) > 0.1] 
plt.figure(figsize=(40, 40))
sns.heatmap(cont_tab.T, annot=True, cmap="YlGnBu", fmt=".1f", cbar = False)

In [ ]:
### Select a cluster to subcluster
cluster_to_sub = 15


adata_subcluster = adata[adata.obs[clusters_to_use] == cluster_to_sub]
adata_subcluster.obs[clusters_to_use].sample() , adata_subcluster.shape

In [ ]:
sc.pp.pca(adata_subcluster)
sc.pp.neighbors(adata_subcluster)
sc.tl.umap(adata_subcluster)

In [ ]:
# extract pca coordinates
X_pca = adata_subcluster.obsm['X_pca'] 

### Kmeans clustering
### You can choose the number of clusters by uncommenting n_clusters option
# kmeans = KMeans(#n_clusters=4,
#                 random_state=0).fit(X_pca) 
# adata_subcluster.obs['kmeans'] = kmeans.labels_.astype(str)

sc.tl.leiden(adata_subcluster, resolution = 0.2)

In [ ]:
clustering_method = 'leiden'

In [ ]:
from matplotlib.pyplot import rc_context
with rc_context({"figure.figsize": (5, 5)}):
    sc.pl.umap(
        adata_subcluster,
        color=clustering_method,
        add_outline=False,
        legend_loc="on data",
        legend_fontsize=12,
        legend_fontoutline=2,
        frameon=False,
        palette="tab20",
    )
# sc.pl.pca(adata_subcluster,
#          color=clustering_method,
#          palette="tab20",
#          )

In [ ]:
### Number of cells per clusters
max_clust = len(adata_subcluster.obs[clustering_method].unique())
for i in range(0, max_clust):
    count = adata_subcluster.obs[clustering_method].value_counts().iloc[i]
    print(f"Cluster {i} : {count} cells")

# adata_subcluster.obs['leiden'].sample(10)

In [ ]:
### Correlation map of subclusters
cont_tab = pd.crosstab(adata_subcluster.obs[clustering_method], adata_subcluster.obs['mmc:class_name'], normalize="index")
plt.figure(figsize=(5,5))
sns.heatmap(cont_tab.T, annot=True, cmap="YlGnBu", fmt=".1f", cbar=False)

In [ ]:
### Correlation map of subclusters
cont_tab = pd.crosstab(adata_subcluster.obs[clustering_method], adata_subcluster.obs['mmc:subclass_name'], normalize="index")
cont_tab = cont_tab.loc[:, cont_tab.sum(axis=0) > 0.05]
plt.figure(figsize=(5, 8))
sns.heatmap(cont_tab.T, annot=True, cmap="YlGnBu", fmt=".1f", cbar=False) 

In [ ]:
da.cluster_plot(adata_subcluster,
                name_dir,
                cluster_to_use = clustering_method,
                cmap_="hls"
                )

In [ ]:
adata_subcluster.obs['new_cluster'] = clustering_method
adata_subcluster.obs['new_cluster2'] = adata_subcluster.obs[clusters_to_use].astype("str") + '.' + adata_subcluster.obs[clustering_method].astype("str")
adata_subcluster.obs[['cell_id','new_cluster2']].sample(2)

In [ ]:
# Use this dictionnary to rename ['cell_type_final'] with the new appropriate cell type for the subcluster. Follow the format. One subcluster at the time.
rename_subclass = {
    f'{cluster_to_sub}.0'  :'',
    f'{cluster_to_sub}.1'   :'',
    f'{cluster_to_sub}.2'   :'',
    f'{cluster_to_sub}.3'   :'',
    f'{cluster_to_sub}.4'   :'',
    f'{cluster_to_sub}.5'   :'',
    f'{cluster_to_sub}.6'   :'',
    f'{cluster_to_sub}.7'   :'',
    f'{cluster_to_sub}.8'   :'',
    f'{cluster_to_sub}.9'   :'',
    f'{cluster_to_sub}.10'  :'',
    f'{cluster_to_sub}.11'  :'',
    f'{cluster_to_sub}.12'  :'',
    f'{cluster_to_sub}.13'  :'',
    f'{cluster_to_sub}.14'  :'',
    f'{cluster_to_sub}.15'  :'',
    f'{cluster_to_sub}.16'  :'',
    f'{cluster_to_sub}.17'  :'',
    }

adata_subcluster.obs['cell_type_final'] = adata_subcluster.obs['new_cluster2'].map(rename_subclass)

# Create a dictionary to map old values to new values
mapping_dict = dict(zip(adata_subcluster.obs['cell_id'], adata_subcluster.obs['cell_type_final']))

# Use .map() function to rename cell contents in 'col1' based on mapping dictionary
adata.obs['cell_type_final'] = adata.obs.apply(lambda x: mapping_dict[x['cell_id']] if x['cell_id'] in mapping_dict else x['cell_type_final'],axis = 1)

In [ ]:
adata.obs["cell_type_final"].sample(5)


In [ ]:
adata.obs['cell_type_final'].value_counts().sort_values()

In [ ]:
adata = adata[adata.obs['cell_type_final'] != 'Undefined']

<font size="6"><span style="color:red">From here, go back to process the other cluster if needed </span></font>

## Save File

In [ ]:
all_cell_type = adata.obs['cell_type_final'].unique()
list_cell_nb = range(0, len(all_cell_type))
mapping_dict = dict(zip(all_cell_type,list_cell_nb))
adata.obs['cell_type_newnum_final'] = adata.obs['cell_type_final'].map(mapping_dict)
mapping_dict

In [ ]:
adata.obs['cell_type_final'].value_counts().sort_values()

In [ ]:
if 'leiden_colors' in adata.obs:
    adata.obs = adata.obs.drop(columns=['leiden_colors'])

adata.write(f"{dir_processed}/h5ad/{name_dir}/{name_dir}_annotated.h5ad.gz", compression='gzip')
adata.obs.to_parquet(f"{dir_processed}/csv/{name_dir}/{name_dir}_annotated.parquet")

In [ ]:
adata = sc.read_h5ad(f"{dir_processed}/h5ad/{name_dir}/{name_dir}_annotated.h5ad.gz")

# Annotation (other tissues)

## Initial annotation

In [ ]:
# sc.tl.leiden(adata, resolution=0.2, flavor = 'leidenalg')

In [ ]:
cluster_to_use = "leiden"

adata.obs[cluster_to_use] = adata.obs[cluster_to_use].astype(str)

sc.tl.rank_genes_groups(adata, groupby=cluster_to_use, method="wilcoxon", tie_correct = True)

print('Ranking done')

In [ ]:
sc.tl.dendrogram(adata, groupby=cluster_to_use)
sc.pl.rank_genes_groups_dotplot(adata,
                                groupby=cluster_to_use,
                                values_to_plot="scores",
                                # min_logfoldchange= 0.26,
                                # standard_scale="var",
                                n_genes=5,
                                dendrogram = False
                                )

In [ ]:
adata.obs.groupby('sample')['leiden'].value_counts().sort_index()

In [ ]:
cluster_to_use = 'leiden'

da.cluster_plot(adata,
                name_dir,
                cluster_to_use=cluster_to_use,
                cluster_to_map=['all']
                )

In [ ]:
adata.obs['leiden'] = adata.obs['leiden'].astype(str)
annotation_dict ={
    "0" : "",
    "1" : "",
    "2" : "",
    "3" : "",
    "4" : "",
    "5" : "",
    "6" : "",
    "7" : "",
    "8" : "",
    "9" : "",
    "10" : "",
    "11" : "",
    "12" : "",
    "13" : "",


}

adata.obs['cell_type_final'] = adata.obs['leiden'].map(annotation_dict)

# Generate new numbering base on unique 'cell type'
all_cell_type = adata.obs['cell_type_final'].unique()
list_cell_nb = range(0, len(all_cell_type))
mapping_dict = dict(zip(all_cell_type,list_cell_nb))
adata.obs['cell_type_newnum'] = adata.obs['cell_type_final'].map(mapping_dict)
mapping_dict

## Subclustering

In [ ]:
clusters_to_use = 'cell_type_newnum_auto_sub'

### Select a cluster to subcluster
cluster_to_sub = "7"

adata_subcluster = adata[adata.obs[clusters_to_use] == cluster_to_sub]
adata_subcluster.obs[clusters_to_use].sample() , adata_subcluster.shape

In [ ]:
sc.pp.pca(adata_subcluster)
sc.pp.neighbors(adata_subcluster)
sc.tl.umap(adata_subcluster)

clustering_method = 'leiden'

# extract pca coordinates
X_pca = adata_subcluster.obsm['X_pca'] 

### Kmeans clustering
### You can choose the number of clusters by uncommenting n_clusters option
if clustering_method == 'kmeans':
    kmeans = KMeans(#n_clusters=4,
                    random_state=0).fit(X_pca) 
    adata_subcluster.obs['kmeans'] = kmeans.labels_.astype(str)
elif clustering_method == "leiden":
    sc.tl.leiden(adata_subcluster, resolution = 0.2)

from matplotlib.pyplot import rc_context
with rc_context({"figure.figsize": (5, 5)}):
    sc.pl.umap(
        adata_subcluster,
        color=clustering_method,
        add_outline=False,
        legend_loc="on data",
        legend_fontsize=12,
        legend_fontoutline=2,
        frameon=False,
        palette="tab20",
    )

max_clust = len(adata_subcluster.obs[clustering_method].unique())
for i in range(0, max_clust):
    count = adata_subcluster.obs[clustering_method].value_counts().iloc[i]
    print(f"Cluster {i} : {count} cells")

da.cluster_plot(adata_subcluster,
                name_dir,
                cluster_to_use=clustering_method,
                cluster_to_map=['all'])

In [ ]:
adata_subcluster.obs['new_cluster'] = clustering_method
adata_subcluster.obs['new_cluster2'] = adata_subcluster.obs[clusters_to_use].astype("str") + '.' + adata_subcluster.obs[clustering_method].astype("str")
adata_subcluster.obs[['cell_id','new_cluster2']].sample(2)

In [ ]:
# Use this dictionnary to rename ['cell type'] with the new appropriate cell type for the subcluster. Follow the format. One subcluster at the time.
rename_subclass = {
    f'{cluster_to_sub}.0'  :'',
    f'{cluster_to_sub}.1'   :'',
    f'{cluster_to_sub}.2'   :'',
    f'{cluster_to_sub}.3'   :'',
    f'{cluster_to_sub}.4'   :'',
    f'{cluster_to_sub}.5'   :'',
    f'{cluster_to_sub}.6'   :'',
    f'{cluster_to_sub}.7'   :'',
    f'{cluster_to_sub}.8'   :'',
    f'{cluster_to_sub}.9'   :'',
    f'{cluster_to_sub}.10'  :'',
    f'{cluster_to_sub}.11'  :'',
    f'{cluster_to_sub}.12'  :'',
    f'{cluster_to_sub}.13'  :'',
    f'{cluster_to_sub}.14'  :'',
    f'{cluster_to_sub}.15'  :'',
    f'{cluster_to_sub}.16'  :'',
    f'{cluster_to_sub}.17'  :'',
    }

adata_subcluster.obs['cell_type_final'] = adata_subcluster.obs['new_cluster2'].map(rename_subclass)

# Create a dictionary to map old values to new values
mapping_dict = dict(zip(adata_subcluster.obs['cell_id'], adata_subcluster.obs['cell_type_final']))

# Use .map() function to rename cell contents in 'col1' based on mapping dictionary
adata.obs['cell_type_final'] = adata.obs.apply(lambda x: mapping_dict[x['cell_id']] if x['cell_id'] in mapping_dict else x['cell_type_final'],axis = 1)

## Save file

In [ ]:
all_cell_type = adata.obs['cell_type_final'].unique()
list_cell_nb = range(0, len(all_cell_type))
mapping_dict = dict(zip(all_cell_type,list_cell_nb))
adata.obs['cell_type_newnum_final'] = adata.obs['cell_type_final'].map(mapping_dict)
mapping_dict

In [ ]:
adata.obs['cell_type_final'].value_counts().sort_values()

In [ ]:
if 'leiden_colors' in adata.obs:
    adata.obs = adata.obs.drop(columns=['leiden_colors'])

adata.write(f"{dir_processed}/h5ad/{name_dir}/{name_dir}_annotated.h5ad.gz", compression='gzip')
adata.obs.to_parquet(f"{dir_processed}/csv/{name_dir}/{name_dir}_annotated.parquet")


In [ ]:
### Load file
adata = sc.read_h5ad(f"{dir_processed}/h5ad/{name_dir}/{name_dir}_annotated.h5ad.gz")

# Automap (brain)

## Data pre-processing

In [ ]:
# testdf = pd.read_csv('Xenium-data-coordinates-CTX.csv')
testdf = pd.read_parquet(f"{dir_processed}/csv/{name_dir}/{name_dir}_MMC_Banksy_annotated_combined.parquet")
testdf.shape

testdf = testdf.filter(['cell_id','run','sample','x_centroid','y_centroid','cell_type_final','cell_type_newnum_final'], axis=1)
### Only keep necessary columns

In [ ]:
print(*testdf['cell_type_final'].unique(), sep='","')

In [ ]:
# Simplify names for mapping
rename_subclass = {
 "Lamp5_Gaba":"Undefined",
 "Pvalb_Gaba":"Undefined",
 "Sst_Gaba":"Undefined",
 "Vip_Gaba":"Undefined",
 "SST_Gaba":"Undefined",
 "Tanycyte":"Undefined",
 "Microglia":"Undefined",
 "Oligodendrocyte":"Undefined",
 "Astrocyte":"Undefined",
 "Ependymal":"Ependymal",
 "VLMC":"VLMC",
 "ABC":"Undefined",
 "Choroid":"Choroid",
 "Pericyte":"Undefined",
 "OPC":"Undefined",
 "Endothelial":"Undefined",
"SI_Gaba":"SI",
 "MEA_Glut":"AMY",
 "PVH_SO_PVa_Glut":"PVH",
 "LHA_Glut":"LHA",
 "BST_MPN_Gaba":"BST",
 "AHN_Glut":"AHN",
 "PVH_SO_Glut":"PVH",
 "SCH_Gaba":"SCH",
 "STR_PAL_Gaba":"STR",

 "L2_3_IT_PIR_ENTl_Glut":"CTX",
 "STR_Gaba":"STR",
 "STR_D2_Gaba":"STR",
 "L6b_CTX_Glut":"CTX",
 "L2_3_IT_CTX_Glut":"CTX",
 "L6_IT_CTX_Glut":"CTX",
 "L5_ET_CTX_Glut":"CTX",
 "STR_D1_Gaba":"STR",
 "LHA_AHN_PVH_Glut":"LHA",
 "COAa_PAA_MEA_Glut":"AMY",

 "LSX_Gaba":"LSX",
 "NLOT_Glut":"CTX",

 "L4_5_IT_CTX_Glut":"CTX",

 "CLA_EPd_CTX_Glut":"CTX",
 "L6_CT_CTX_Glut":"CTX",
 "PAL_STR_Gaba_Chol":"STR",
 "BST_po_Glut":"BST",
 "PVT_Glut":"PVT",
 "PT_Glut":"PT",
 "RE_Glut":"RE",
 "OB_STR_CTX_Inh_IMN":"Undefined",
 "L5_NP_CTX_Glut":"CTX",

 "RT_ZI_Gaba":"RT",
 "CA3_Glut":"HIPP",
 "CEA_AAA_BST_Gaba":"AMY",
 "TRS_BAC_Glut":"TRS",

 "LSX_Glut":"LSX",
 "STR_GABA":"STR",
 "DG_PIR_Ex_IMN":"HIPP",

 "L2_3_IT_RSP_Glut":"CTX",
 "AV_Glut":"AV",
 "TH_Glut":"TH",
 "ATN_Glut":"ATN",
 "PVR_Gaba":"PVR",
 "PVR_Glut":"PVR",

 "DG_Glut":"HIPP",

 "MH_Glut":"MH",
 "L4_RSP_ACA_Glut":"CTX",
 "AD_Glut":"AD",
 "LH_Glut":"LH",


 "CA1_ProS_Glut":"HIPP",

 "HPF_CR_Glut":"Undefined",
 "CA2_FC_IG_Glut":"HIPP",


'VMH_Glut':"VMH",
'ARH_Gaba':"ARH",
'ZI_Gaba':"ZI",
'DMH_Gaba':"DMH",
'ARH_Glut':"ARH",
       'SC_Glut':"SC",
       'DMH_Glut':"DMH",
       'Astro_NT':"Undefined",
       'PRC_PAG_Glut':'MB',
       'PF_Glut':"PF",
       'LA_BLA_BMA_PA_Glut':'AMY',
       'Astro_TE': "Undefined",
       'MEA_COA_BMA_Glut':"AMY",
       'MEA_BST_Gaba':"AMY",
       'RT_Gaba':"TH",
       'STN_Glut':"TH",
       'L2_3_IT_ENT_Glut':"CTX",
       'L5_6_IT_TPE_ENT_Glut':"CTX",
       'Sncg_Gaba':"CTX",
       'IT_EP_CLA_Glut':"CTX",
       'CEA_BST_Gaba':"AMY",
       'STRv_Gaba':"STR",
        'L5_IT_CTX_Glut':"CTX",
        'L6b_EPd_Glut':"CTX",
       'SPA_SPF_Glut':'TH',
       'CA1_Glut':'HIPP',
       'HPF_CR_ Glut':"HIPP",
       'PRT_Gaba':"MB",
       'PM_Glut':"PM",
       'MM_Glut':"MM",
       'CA2_Glut':'HIPP',
       'SC_Gaba':"SC",
       'FC_Glut':'HIPP',
       'RHP_COA_Gaba':'AMY',
       'LGv_Gaba':"TH",
       'SNc_VTA_RAmb_Dopa':"SN",
       'Sub_Glut':"HIPP",
       'Pineal_Glut': "Undefined",
       'SNr_Gaba':"SN",
       'NP_Glut':"MB",
       'Hypendymal' : "Undefined",
       'ND_INC_Glut':"MB",
       'APN_Glut':"MB",

       'ADP_MPO_Glut':"MPO",
       'MS_SF_Glut':'Undefined',
       'NDB_SI_MA_STRv_Gaba':"STR",
       'AHN_RCH_LHA_Glut':"LHA",
       'SI_MPO_LPO_Gaba':'MPO',
       'MEA_Gaba':"MEA",
       'SI_MA_LPO_LHA_Glut':"LHA",
       'Sst_Chodl_Gaba':'Undefined',
       'GPe_SI_Gaba':"STR",
       'RE_Xi_Glut':"RE",
       'IA_Gaba':'Undefined',
       'PVT_PT_Glut':"PVT",
       'OB_out_Gaba':'Undefined',
 }

testdf['cell_type_final'] = testdf['cell_type_final'].apply(lambda x: rename_subclass[x] if x in rename_subclass else x)
testdf = testdf[testdf['cell_type_final'] != 'Undefined']
testdf = testdf[testdf['cell_type_final'] != 'undefined']
testdf['cell_type_final'].unique()

In [ ]:
sample_ids = testdf['sample'].unique()
sample_ids

In [ ]:
from module.automap import knn_mst_clustering
from module.automap import assign_chunk_labels
from module.automap import fill_any_chunks,fill_empty_chunks
from module.automap import grid_to_geojson_with_scaling
import json
from IPython.display import clear_output


count = 0
for sample_to_map in sample_ids:

    count +=1
    print(f'{datetime.now()}: automap of {sample_to_map} ({count}/{len(samples_ids)})')

    df = testdf[testdf['sample']==sample_to_map]
    all_cell_type = df['cell_type_final'].unique()
    list_cell_nb = range(0, len(all_cell_type))
    mapping_dict = dict(zip(all_cell_type,list_cell_nb))
    df['cell_type_newnum_final'] = df['cell_type_final'].map(mapping_dict)

    # Multiply centroid coordinates by 10 to convert to pixel coordinates
    df['x_pixel'] = df['x_centroid'] * 10
    df['y_pixel'] = df['y_centroid'] * 10

    ### Generate a color palette for the clusters - to make color stay consistent across samples
    df['cell_type_newnum_final'] = df['cell_type_newnum_final'].astype(str)

    ### Establish neighbors classifiers

    K = 10
    X = df[['x_pixel', 'y_pixel']].values
    y = df['cell_type_newnum_final'].astype(int).values
    knn = KNeighborsClassifier(n_neighbors=K)
    knn.fit(X, y)
    df['knn_celltype'] = knn.predict(X)

    data = df
    valid_celltypes = data['knn_celltype'].value_counts()[data['knn_celltype'].value_counts() > 100].index
    data = data[data['knn_celltype'].isin(valid_celltypes)]
    unique_cell_types = sorted(data['knn_celltype'].unique())

    for cell_type in unique_cell_types:
        print(f"Processing cell type {cell_type}...")
        knn_mst_clustering(data, cell_type=cell_type, k=10)

    # Save the updated dataframe with cluster labels
    # data.to_csv('Xenium-data-coordinates-CTX-labeled.csv', index=False)
    print("Data with cluster labels saved.")

    data['knn_celltype'].value_counts()
    data['cluster_label'].isna().sum()

    # Get the value counts of each category
    value_counts = data['cluster_label'].value_counts()

    # Create a mapping based on the rank of value counts
    mapping_drg = {label: idx for idx, label in enumerate(value_counts.index, 1)}

    # Map the categorical strings to numbers based on the value counts
    data['cluster_label_numeric'] = data['cluster_label'].map(mapping_drg)
    data['cluster_label_numeric'].unique()
    data = data.dropna(subset=['cluster_label_numeric'])
    data['cluster_label_numeric'].isna().sum()

    ### Pixelize section and assign label
    chunk_size = 400  # Example chunk size
    threshold = 0.5  # Example threshold
    grid, x_bins, y_bins = assign_chunk_labels(data, chunk_size, threshold)


    filled_grid, x_bins, y_bins = fill_empty_chunks(grid, x_bins, y_bins, 4)
    filled_grid, x_bins, y_bins = fill_empty_chunks(filled_grid, x_bins, y_bins, 3)
    filled_grid, x_bins, y_bins = fill_empty_chunks(filled_grid, x_bins, y_bins, 4)
    filled_grid, x_bins, y_bins = fill_empty_chunks(filled_grid, x_bins, y_bins, 3)
    filled_grid, x_bins, y_bins = fill_any_chunks(filled_grid, x_bins, y_bins,4)
    filled_grid, x_bins, y_bins = fill_empty_chunks(filled_grid, x_bins, y_bins, 3)
    filled_grid, x_bins, y_bins = fill_empty_chunks(filled_grid, x_bins, y_bins, 2)
    filled_grid, x_bins, y_bins = fill_empty_chunks(filled_grid, x_bins, y_bins, 2)
    filled_grid, x_bins, y_bins = fill_empty_chunks(filled_grid, x_bins, y_bins, 3)
    filled_grid, x_bins, y_bins = fill_any_chunks(filled_grid, x_bins, y_bins,4)
    filled_grid, x_bins, y_bins = fill_any_chunks(filled_grid, x_bins, y_bins,3)
    filled_grid, x_bins, y_bins = fill_any_chunks(filled_grid, x_bins, y_bins,3)

    # Create the mapping as before
    mapping = data.groupby('knn_celltype')['cell_type_final'].agg(lambda x: x.value_counts().idxmax()).to_dict()

    # Create a new column 'knn_celltype_label' using the mapping
    data['knn_celltype_label'] = data['knn_celltype'].map(mapping)
    data['cell_type_newnum_final'] = data['cell_type_newnum_final'].astype(int)

    # Display the dataframe to verify
    display(data[['knn_celltype', 'knn_celltype_label']].sample(3))

    ### Export geojson
    value_counts = data[['knn_celltype', 'cell_type_final']].value_counts()
    value_counts_df = value_counts.reset_index(name='count')
    most_common_pairs = value_counts_df.groupby('knn_celltype').apply(lambda x: x.nlargest(1, 'count'))
    celltype_to_newnum = most_common_pairs.set_index('knn_celltype')['cell_type_final'].to_dict()
    celltype_to_newnum = {key + 1: value for key, value in celltype_to_newnum.items()}

    geojson_data = grid_to_geojson_with_scaling(filled_grid, x_bins, y_bins, celltype_to_newnum, scale_factor=10)
    clear_output()
    # Save the GeoJSON to a file
    with open(f'{dir_processed}/coordinates/Region_prediction/Xenium-data-coordinates-filtered_{sample_to_map}.geojson', 'w') as f:
        json.dump(geojson_data, f, indent=2)


print(f'{datetime.now()}: automap done')




## Match cells with automatically generated regions

### Data pre-processing

In [ ]:
adata = sc.read_h5ad(f"{dir_processed}/h5ad/{name_dir}/{name_dir}_MMC_Banksy_annotated_combined.h5ad.gz")

In [ ]:
samples_ids = adata.obs['sample'].unique()
samples_ids

In [ ]:
count = 0
combine_dict = {}

for sample_to_map in samples_ids:
    count += 1
    print(f'{datetime.now()}: mapping of {sample_to_map} ({count}/{len(samples_ids)})')
    geojson  = gpd.read_file(f'{dir_processed}/coordinates/Region_prediction/Xenium-data-coordinates-filtered_{sample_to_map}.geojson')
    geojson['geometry'][1]
    adata_sub = adata[adata.obs['sample'] == sample_to_map]

    ### Cell mapping
    centroid_gdp = gpd.GeoDataFrame(adata_sub.obs, geometry=gpd.points_from_xy(adata_sub.obs.x_centroid, adata_sub.obs.y_centroid))
    centroid_gdp.index.name = None
    centroid_gdp.crs = 'EPSG:4326'
    matched_cells = gpd.sjoin(centroid_gdp, geojson, predicate='within', how='left')

    mapping_dict_reg = dict(zip(matched_cells['cell_id'], matched_cells['cell_type_newnum_final_right']))
    adata_sub.obs['region_automap'] = adata_sub.obs['cell_id'].map(mapping_dict_reg)

    all_cell_type = adata_sub.obs['region_automap'].unique()
    list_cell_nb = range(0, len(all_cell_type))
    mapping_dict = dict(zip(all_cell_type,list_cell_nb))
    adata_sub.obs['region_automap_newnum'] = adata_sub.obs['region_automap'].map(mapping_dict)

    mapping_dict_reg = dict(zip(adata_sub.obs['cell_id'], adata_sub.obs['region_automap']))
    combine_dict.update(mapping_dict_reg)
    
print(f'{datetime.now()}: dict done')
adata.obs['region_automap_name'] = adata.obs['cell_id'].map(combine_dict)

if 'leiden_colors' in adata.obs:
    adata.obs = adata.obs.drop(columns=['leiden_colors'])

adata.write(f"{dir_processed}/h5ad/{name_dir}/{name_dir}_MMC_Banksy_annotated_automap.h5ad.gz", compression = 'gzip')

# adata.obs.to_csv(f"{dir_processed}/csv/{name_dir}/{name_dir}_MMC_Banksy_annotated_automap.csv.gz",
#          compression={'method': 'gzip'})


In [ ]:
all_cell_type = adata.obs['region_automap_name'].unique()
list_cell_nb = range(0, len(all_cell_type))
mapping_dict = dict(zip(all_cell_type,list_cell_nb))
adata.obs['region_automap_num'] = adata.obs['region_automap_name'].map(mapping_dict)
mapping_dict

## Check regions mapping

In [ ]:
da.cluster_plot(adata,
             name_dir,
             cluster_to_use = 'region_automap_num',
             cluster_to_map = ['all'],
             cmap_ = 'tab20b',
            #  save_plot = True,
            )

## Alternative Automap : WIP

In [ ]:
testdf = pd.DataFrame(data=adata.X.toarray(), index=adata.obs_names, columns=adata.var_names)

from module.xenium_preprocessing import add_annotations
testdf = add_annotations(adata,testdf)

# testdf = testdf.filter(['cell_id','sample','x_centroid','y_centroid','cell_type_final','cell_type_newnum_final','region_automap_name'], axis=1)
### Only keep necessary columns

In [ ]:
anno = 'x_centroid'
dict_temp = dict(zip(adata.obs['cell_id'], adata.obs[anno]))
testdf[anno] = testdf['cell_id'].map(dict_temp)
anno = 'y_centroid'
dict_temp = dict(zip(adata.obs['cell_id'], adata.obs[anno]))
testdf[anno] = testdf['cell_id'].map(dict_temp)

In [ ]:
testdf = testdf[testdf['region_automap_name']=='CTX']
testdf.sample()

In [ ]:
rename_subclass = {'Astro TE': 'Undefined',
 'Sst Gaba': 'Sst Gaba',
 'L2 3 IT PIR ENTl Glut': 'L2 3',
 'L6b CTX Glut': 'L6',
 'L4 5 IT CTX Glut': 'L4 5',
 'L6 IT CTX Glut': 'L6',
 'L5 ET CTX Glut': 'L5',
 'Pvalb Gaba': 'Pvalb Gaba',
 'Oligodendrocyte': 'Undefined',
 'DG Glut': 'Undefined',
 'COAa PAA MEA Glut': 'COA',
 'Lamp5 Gaba': 'Lamp5 Gaba',
 'Microglia': 'Undefined',
 'OPC': 'Undefined',
 'Pericyte': 'Undefined',
 'CLA EPd CTX Glut': 'CLA',
 'L6 CT CTX Glut': 'L6',
 'L2 3 PIR ENTl Glut': 'L2 3',
 'L5 NP CTX Glut': 'L5',
 'Sncg Gaba': 'Sncg Gaba',
 'Vip Gaba': 'Vip Gaba',
 'HY Glut': 'Undefined',
 'VLMC': 'Undefined',
 'ABC': 'Undefined',
 'Endothelial': 'Undefined',
 'STR PAL Gaba': 'Undefined',
 'STR D2 Gaba': 'Undefined',
 'OB STR CTX IMN': 'Undefined',
 'SMC': 'Undefined',
 'STR Gaba': 'Undefined',
 'PVT PT Glut': 'Undefined',
 'CA2 FC IG Glut': 'Undefined',
 'LSX Gaba': 'Undefined',
 'TRS BAC Glut': 'Undefined',
 'NLOT Glut': 'NLOT Glut',
 'STR D1 Gaba': 'Undefined',
 'PAL STR Gaba Chol': 'Undefined',
 'L2 3 IT RSP Glut': 'L2 3',
 'CA3 Glut': 'Undefined',
 'SCH Gaba': 'Undefined',
 'TH Glut': 'Undefined',
 'Ependymal': 'Undefined',
 'RT ZI Gaba': 'Undefined',
 'CA1 ProS Glut': 'Undefined',
 'Choroid': 'Undefined',
 'SUndefined':'Undefined'}

In [ ]:
testdf['cell_type_final'] = testdf['cell_type_final'].apply(lambda x: rename_subclass[x] if x in rename_subclass else x)
testdf = testdf[testdf['cell_type_final'] != 'Undefined']
testdf = testdf[testdf['cell_type_final'] != 'undefined']
testdf['cell_type_final'].unique()

In [ ]:
from module.automap import knn_mst_clustering
from module.automap import assign_chunk_labels
from module.automap import fill_any_chunks,fill_empty_chunks
from module.automap import grid_to_geojson_with_scaling
import json
from IPython.display import clear_output


count = 0
for sample_to_map in sample_ids:

    count +=1
    print(f'{datetime.now()}: automap of {sample_to_map} ({count}/{len(samples_ids)})')

    df = testdf[testdf['sample']==sample_to_map]
    all_cell_type = df['cell_type_final'].unique()
    list_cell_nb = range(0, len(all_cell_type))
    mapping_dict = dict(zip(all_cell_type,list_cell_nb))
    df['cell_type_newnum_final'] = df['cell_type_final'].map(mapping_dict)

    # Multiply centroid coordinates by 10 to convert to pixel coordinates
    df['x_pixel'] = df['x_centroid'] * 10
    df['y_pixel'] = df['y_centroid'] * 10

    ### Generate a color palette for the clusters - to make color stay consistent across samples
    df['cell_type_newnum_final'] = df['cell_type_newnum_final'].astype(str)

    ### Establish neighbors classifiers

    K = 10
    X = df[['x_pixel', 'y_pixel']].values
    y = df['cell_type_newnum_final'].astype(int).values
    knn = KNeighborsClassifier(n_neighbors=K)
    knn.fit(X, y)
    df['knn_celltype'] = knn.predict(X)

    data = df
    valid_celltypes = data['knn_celltype'].value_counts()[data['knn_celltype'].value_counts() > 100].index
    data = data[data['knn_celltype'].isin(valid_celltypes)]
    unique_cell_types = sorted(data['knn_celltype'].unique())

    for cell_type in unique_cell_types:
        print(f"Processing cell type {cell_type}...")
        knn_mst_clustering(data, cell_type=cell_type, k=10)

    # Save the updated dataframe with cluster labels
    # data.to_csv('Xenium-data-coordinates-CTX-labeled.csv', index=False)
    print("Data with cluster labels saved.")

    data['knn_celltype'].value_counts()
    data['cluster_label'].isna().sum()

    # Get the value counts of each category
    value_counts = data['cluster_label'].value_counts()

    # Create a mapping based on the rank of value counts
    mapping_drg = {label: idx for idx, label in enumerate(value_counts.index, 1)}

    # Map the categorical strings to numbers based on the value counts
    data['cluster_label_numeric'] = data['cluster_label'].map(mapping_drg)
    data['cluster_label_numeric'].unique()
    data = data.dropna(subset=['cluster_label_numeric'])
    data['cluster_label_numeric'].isna().sum()

    ### Pixelize section and assign label
    chunk_size = 400  # Example chunk size
    threshold = 0.5  # Example threshold
    grid, x_bins, y_bins = assign_chunk_labels(data, chunk_size, threshold)


    filled_grid, x_bins, y_bins = fill_empty_chunks(grid, x_bins, y_bins, 4)
    filled_grid, x_bins, y_bins = fill_empty_chunks(filled_grid, x_bins, y_bins, 3)
    filled_grid, x_bins, y_bins = fill_empty_chunks(filled_grid, x_bins, y_bins, 4)
    filled_grid, x_bins, y_bins = fill_empty_chunks(filled_grid, x_bins, y_bins, 3)
    filled_grid, x_bins, y_bins = fill_any_chunks(filled_grid, x_bins, y_bins,4)
    filled_grid, x_bins, y_bins = fill_empty_chunks(filled_grid, x_bins, y_bins, 3)
    filled_grid, x_bins, y_bins = fill_empty_chunks(filled_grid, x_bins, y_bins, 2)
    filled_grid, x_bins, y_bins = fill_empty_chunks(filled_grid, x_bins, y_bins, 2)
    filled_grid, x_bins, y_bins = fill_empty_chunks(filled_grid, x_bins, y_bins, 3)
    filled_grid, x_bins, y_bins = fill_any_chunks(filled_grid, x_bins, y_bins,4)
    filled_grid, x_bins, y_bins = fill_any_chunks(filled_grid, x_bins, y_bins,3)
    filled_grid, x_bins, y_bins = fill_any_chunks(filled_grid, x_bins, y_bins,3)

    # Create the mapping as before
    mapping = data.groupby('knn_celltype')['cell_type_final'].agg(lambda x: x.value_counts().idxmax()).to_dict()

    # Create a new column 'knn_celltype_label' using the mapping
    data['knn_celltype_label'] = data['knn_celltype'].map(mapping)
    data['cell_type_newnum_final'] = data['cell_type_newnum_final'].astype(int)

    # Display the dataframe to verify
    display(data[['knn_celltype', 'knn_celltype_label']].sample(3))

    ### Export geojson
    value_counts = data[['knn_celltype', 'cell_type_final']].value_counts()
    value_counts_df = value_counts.reset_index(name='count')
    most_common_pairs = value_counts_df.groupby('knn_celltype').apply(lambda x: x.nlargest(1, 'count'))
    celltype_to_newnum = most_common_pairs.set_index('knn_celltype')['cell_type_final'].to_dict()
    celltype_to_newnum = {key + 1: value for key, value in celltype_to_newnum.items()}

    geojson_data = grid_to_geojson_with_scaling(filled_grid, x_bins, y_bins, celltype_to_newnum, scale_factor=10)
    clear_output()
    # Save the GeoJSON to a file
    with open(f'{dir_processed}/coordinates/Region_prediction/Xenium-data-coordinates-filtered_{sample_to_map}.geojson', 'w') as f:
        json.dump(geojson_data, f, indent=2)


print(f'{datetime.now()}: automap done')




In [ ]:
count = 0
combine_dict = {}

for sample_to_map in samples_ids:
    count += 1
    print(f'{datetime.now()}: mapping of {sample_to_map} ({count}/{len(samples_ids)})')
    geojson  = gpd.read_file(f'{dir_processed}/coordinates/Region_prediction/Xenium-data-coordinates-filtered_{sample_to_map}.geojson')
    geojson['geometry'][1]
    adata_sub = adata[adata.obs['sample'] == sample_to_map]

    ### Cell mapping
    centroid_gdp = gpd.GeoDataFrame(adata_sub.obs, geometry=gpd.points_from_xy(adata_sub.obs.x_centroid, adata_sub.obs.y_centroid))
    centroid_gdp.index.name = None
    centroid_gdp.crs = 'EPSG:4326'
    matched_cells = gpd.sjoin(centroid_gdp, geojson, predicate='within', how='left')

    mapping_dict_reg = dict(zip(matched_cells['cell_id'], matched_cells['cell_type_newnum_final_right']))
    adata_sub.obs['region_automap'] = adata_sub.obs['cell_id'].map(mapping_dict_reg)

    all_cell_type = adata_sub.obs['region_automap'].unique()
    list_cell_nb = range(0, len(all_cell_type))
    mapping_dict = dict(zip(all_cell_type,list_cell_nb))
    adata_sub.obs['region_automap_newnum'] = adata_sub.obs['region_automap'].map(mapping_dict)

    mapping_dict_reg = dict(zip(adata_sub.obs['cell_id'], adata_sub.obs['region_automap']))
    combine_dict.update(mapping_dict_reg)
    
print(f'{datetime.now()}: dict done')
adata.obs['region_automap_CTX_name'] = adata.obs['cell_id'].map(combine_dict)

# Manual Map


## Data preparation

In [ ]:
adata = sc.read_h5ad(f"{dir_processed}/h5ad/{name_dir}/{name_dir}_annotated_automap.h5ad.gz") 
# adata = sc.read_h5ad(f"{dir_processed}/h5ad/{name_dir}/{name_dir}_final.h5ad.gz") 

In [ ]:
sample_ids = adata.obs['sample'].unique()
sample_ids = sample_ids[0:6]
sample_ids

In [ ]:
### Choose sample to map here:
sample_to_map = sample_ids[0]
sample_to_map
BR_df = pd.read_csv(f"coordinates/XE_selection/{sample_to_map}-RL.csv",comment = '#')

In [ ]:
#Preprocessing the coordinates from Xenium Explorer into usable geojson

for sample_to_map in sample_ids:
    BR_df = pd.read_csv(f"coordinates/XE_selection/{sample_to_map}-RL.csv",comment = '#')
    # Group the dataframe by the "Selection" column
    grouped = BR_df.groupby('Selection')

    # List to hold GeoJSON features
    features = []

    for name, group in grouped:
        # Create a list of coordinates for each region
        coordinates = [(x, y) for x, y in zip(group['X'], group['Y'])]
        if coordinates[0] != coordinates[-1]:
            coordinates.append(coordinates[0])
        
        # Create a GeoJSON polygon for the region
        polygon = geojson.Polygon([coordinates])
        feature = geojson.Feature(geometry=polygon, properties={"region": name})
        features.append(feature)

    # Create a GeoJSON FeatureCollection
    feature_collection = geojson.FeatureCollection(features)

    # Save the GeoJSON FeatureCollection to a file
    with open(f'coordinates/{sample_to_map}_regions_manual.geojson', 'w') as f:
        geojson.dump(feature_collection, f)

    print(f"GeoJSON {sample_to_map} saved")

In [ ]:
for sample_to_map in sample_ids[0:6]:
    regions_df = gpd.read_file(f'{dir_processed}/coordinates/{sample_to_map}_regions_manual.geojson')
    #Region mapping
    adata_region_sub = adata[adata.obs['sample']== sample_to_map]

    centroid_gdp = gpd.GeoDataFrame(adata_region_sub.obs, geometry=gpd.points_from_xy(adata_region_sub.obs.x_centroid, adata_region_sub.obs.y_centroid))
    centroid_gdp.index.name = None
    centroid_gdp.crs = 'EPSG:4326'
    matched_cells = gpd.sjoin(centroid_gdp, regions_df, predicate='within', how='left')
    mapping_dict_reg = dict(zip(matched_cells['cell_id'], matched_cells['region']))
    adata_region_sub.obs['region_manual'] = adata_region_sub.obs['cell_id'].map(mapping_dict_reg)

    all_cell_type = adata_region_sub.obs['region_manual'].unique()
    list_cell_nb = range(0, len(all_cell_type))
    mapping_dict = dict(zip(all_cell_type,list_cell_nb))
    adata_region_sub.obs['region_manual_newnum'] = adata_region_sub.obs['region_manual'].map(mapping_dict)
    mapping_dict_reg = dict(zip(adata_region_sub.obs['cell_id'], adata_region_sub.obs['region_manual']))

    if 'combine_dict_region' not in locals():
        combine_dict_region = {}
        combine_dict_region.update(mapping_dict_reg)   
    else:
        combine_dict_region.update(mapping_dict_reg)

<font size="6"><span style="color:red">From here, go back to process the other samples </span></font>

## Apply combined dictionnary

In [ ]:
### Only run when you are done with all samples automap annotations
adata.obs['region_manual_name'] = adata.obs['cell_id'].map(combine_dict_region)

In [ ]:
adata.obs['region_manual_name'].value_counts()

# Optional annotations

## Cell class (brain, VENG)

In [ ]:
from module.subclustering_Xe import cell_class_annotation

adata = cell_class_annotation(adata)


all_cell_type = adata.obs['cell_class'].unique()
list_cell_nb = range(0, len(all_cell_type))
mapping_dict = dict(zip(all_cell_type,list_cell_nb))
adata.obs['cell_class_newnum'] = adata.obs['cell_class'].map(mapping_dict)
mapping_dict

adata.obs['cell_class'].value_counts()
# adata.obs[adata.obs['cell_class'] == 'Epithelial']['cell_type_final'].value_counts(dropna=True)

In [ ]:
### To add White matter annotation to unlabeled cell region
### Not sure which solution work... to test later

temp = adata.obs['region_automap_name'].cat.add_categories("WM").fillna("WM")
adata.obs['region_automap_name'] = temp
all_cell_type = adata.obs['region_automap_name'].unique()
list_cell_nb = range(0, len(all_cell_type))
mapping_dict = dict(zip(all_cell_type,list_cell_nb))
adata.obs['region_automap_num'] = adata.obs['region_automap_name'].map(mapping_dict)
mapping_dict

## Neurotransmitter

In [ ]:
dict_neurotrans = {}

for cell in adata.obs['cell_type_final'].unique():
    if cell.split('_')[-1] == 'Gaba':
        dict_neurotrans[cell] = 'Gaba'
    elif cell.split('_')[-1] == 'Glut':
        dict_neurotrans[cell] = 'Glutamate'
    elif cell.split('_')[-1] == 'Chol':
        dict_neurotrans[cell] = 'Acetylcholine'
    elif cell.split('_')[-1] == 'Dopa':
        dict_neurotrans[cell] = 'Dopamine'
    else:
        dict_neurotrans[cell] = 'Non-neuronal'

In [ ]:
adata.obs['Neurotransmitter'] = adata.obs['cell_type_final'].map(dict_neurotrans)
adata.obs['Neurotransmitter'].value_counts()

## Number of clock genes per cell

In [ ]:
from module.subclustering_Xe import circascore_annot

df = pd.read_parquet(f"{dir_processed}/csv/{name_dir}/{name_dir}_normalized_counts.parquet")

adata = circascore_annot(adata,df)

In [ ]:
grouped = adata.obs.groupby('cell_type_final')['circascore'].value_counts(normalize=True)

In [ ]:
df.columns

In [ ]:
df.groupby('cell_type_final')['Ruvbl1'].mean()

In [ ]:
for cell in adata.obs['cell_type_final'].unique():
    plt.plot(grouped[cell].values, label = cell)

plt.legend(loc=(1,0.9))
plt.grid()
plt.yscale('log')

## ZT

In [ ]:
### If ZT is in the sample name
adata.obs['ZT'] = adata.obs['sample'].map(lambda name: name.split('-')[-1])

In [ ]:
### Other case
ZT_dict = {
    '3159-1':'ZT17','2670-1':'ZT5','3159-2':'ZT17','3159-3':'ZT17','3159-4':'ZT17','2505-1':'ZT5','2505-2':'ZT5',
'3160-1':'ZT17','3160-2':'ZT17','3161-1':'ZT17','3161-2':'ZT17','3161-3':'ZT17',
}
adata.obs['ZT'] = adata.obs['sample'].map(ZT_dict)

In [ ]:
adata.obs['ZT'].value_counts()

## Genotype

In [ ]:
geno_dict = {'3159-1':'WT','2670-1':'WT','3159-2':'WT','3159-3':'WT','3159-4':'WT',
             '2505-1':'APP','2505-2':'APP','3160-1':'APP','3160-2':'APP','3161-1':'APP','3161-2':'APP','3161-3':'APP',
}

adata.obs['Genotype'] = adata.obs['sample'].map(geno_dict)

## Plaques

In [ ]:
adata.obs.sample(1)

In [ ]:
sample_ids = ["3161-1",'3161-2',"3161-3"]

area_list = []
ctx_list = []

for sample_to_map in sample_ids:
    plaque_gpd = gpd.read_file(f'{dir_processed}/coordinates/plaques/{sample_to_map}_plaques_c.geojson')
    plaque_gpd['area'] = plaque_gpd.area
    area_list.append(plaque_gpd['area'].sum())
    ctx_gpd = gpd.read_file(f'{dir_processed}/coordinates/Region_prediction/Xenium-data-coordinates-filtered_{sample_to_map}.geojson')
    ctx_gpd['area'] = ctx_gpd.area
    ctx_list.append(ctx_gpd['area'].sum())
sum(area_list) / sum(ctx_list) * 100

In [ ]:
sample_ids = ["3161-1",'3161-2',"3161-3"]

area_list = []

for sample_to_map in sample_ids:
    plaque_gpd = gpd.read_file(f'{dir_processed}/coordinates/plaques/{sample_to_map}_plaques_c.geojson')
    plaque_gpd['area'] = plaque_gpd.area
    area_list.append(plaque_gpd['area'].values)

In [ ]:
plt.figure(figsize=(3,5))
sns.histplot(area_list[0], bins=100, kde=True, color = 'black', stat="proportion")
plt.xlabel('Plaque area (µm²)')
plt.savefig(f'Gallery/2025-11-07/plaque_area_distribution.svg', dpi=300, transparent=True)

In [ ]:
adata.obs['plaque'] = "No"

combine_dict_plaque = dict(zip(adata.obs['cell_id'], adata.obs['plaque']))

for sample_to_map in sample_ids:
    plaque_gpd = gpd.read_file(f'{dir_processed}/coordinates/plaques/{sample_to_map}_plaques_c.geojson')

    adata_region_sub = adata[adata.obs['sample']== sample_to_map]

    centroid_gdp = gpd.GeoDataFrame(adata_region_sub.obs, geometry=gpd.points_from_xy(adata_region_sub.obs.x_centroid, adata_region_sub.obs.y_centroid))
    centroid_gdp.index.name = None
    centroid_gdp.crs = 'EPSG:4326'
    matched_cells = gpd.sjoin(centroid_gdp, plaque_gpd, predicate='within', how='left')
    matched_cells['id'] = matched_cells['id'].fillna("0")
    matched_cells = matched_cells[matched_cells['id']!="0"]

    mapping_dict_reg = matched_cells['cell_id'].values

    for cell in mapping_dict_reg:
        combine_dict_plaque[cell] = "Yes"

adata.obs['plaque'] = adata.obs['cell_id'].map(combine_dict_plaque)
adata.obs['plaque'].value_counts()

In [ ]:
adata.obs[adata.obs['region_automap_name']=="CTX"].groupby('plaque')['cell_type_final'].value_counts(normalize=True)

In [ ]:
sample_ = "3161-3"

temp = adata.obs[(adata.obs['sample'] ==sample_)&(adata.obs['plaque']=="Yes")]

plt.figure(figsize=(15,10))
plt.scatter(adata.obs[(adata.obs['sample'] ==sample_)]["x_centroid"], adata.obs[(adata.obs['sample'] ==sample_)]["y_centroid"])
plt.scatter(temp['x_centroid'], temp['y_centroid'])
plt.tight_layout()

# Final Output

In [ ]:
if 'leiden_colors' in adata.obs:
    adata.obs = adata.obs.drop(columns=['leiden_colors'])

adata.write(f"{dir_processed}/h5ad/{name_dir}/{name_dir}_final.h5ad.gz", compression = 'gzip')

# adata.obs.to_csv(f"{dir_processed}/csv/{name_dir}/{name_dir}_MMC_Banksy_annotated_automap.csv.gz",
#          compression={'method': 'gzip'})

In [ ]:
df = pd.DataFrame(data=adata.X.toarray(), index=adata.obs_names, columns=adata.var_names)
df.shape

from module.xenium_preprocessing import add_annotations

df = add_annotations(adata, df)

# df.to_csv(f"{dir_processed}/csv/{name_dir}/{name_dir}_norm_combined.csv")
df.to_parquet(f"{dir_processed}/csv/{name_dir}/{name_dir}_norm_combined.parquet")

In [ ]:
adata = sc.read_h5ad(f"{dir_processed}/h5ad/{name_dir}/{name_dir}_final.h5ad.gz")

# End of the notebook

Next step : Data vizualization and analysis

[v11C_DataViz_Analysis](./v11C_DataViz_Analysis.ipynb)